# Laboratorio 9 - Deep Learning

- Juan Solís - 23720
- Victor Pérez - 23731
- Diego Flores - 23714
- Nils Muralles - 23727
- Isabella Recinos - 23003

## Task 4

### Task 4.1

Se reconstruyen el VAE con β = 1 (Task 1) y la U-Net condicional (Task 3) con la misma semilla e hiperparámetros. Si existen los checkpoints en `data/` se cargan; si no, se entrenan.

In [1]:
import math, time
from pathlib import Path
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path("data").mkdir(exist_ok=True)

data = datasets.FashionMNIST("data", train=True, download=True)
X01 = data.data.float() / 255.0          # [0,1] para el VAE
Y = data.targets
print("semilla:", SEED, "| device:", device)

semilla: 42 | device: cuda


**VAE β = 1** (misma arquitectura y pérdida de la Task 1.2).

In [2]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.dec = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(), nn.Linear(256, 512), nn.ReLU(),
                                 nn.Linear(512, 784), nn.Sigmoid())

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.fc_mu(h), self.fc_logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)   # z = mu + sigma * eps
        return self.dec(z), mu, logvar

x_train_vae = X01[:-5000].view(-1, 784).to(device)
vae = VAE().to(device)
vae_ckpt = Path("data/task1_vae_beta1.pt")
if vae_ckpt.exists():
    vae.load_state_dict(torch.load(vae_ckpt, map_location=device))
else:
    torch.manual_seed(SEED)
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
    for ep in range(20):
        perm = torch.randperm(len(x_train_vae), device=device)
        for i in range(0, len(x_train_vae), 128):
            xb = x_train_vae[perm[i:i + 128]]
            x_hat, mu, logvar = vae(xb)
            # L_beta = ||x - x_hat||^2 + beta * KL, con beta = 1
            kl = 0.5 * torch.sum(mu**2 + logvar.exp() - 1 - logvar, dim=1)
            loss = (torch.sum((xb - x_hat) ** 2, dim=1) + kl).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    torch.save(vae.state_dict(), vae_ckpt)
vae.eval()
print("VAE listo")

VAE listo


**U-Net condicional y calendario lineal** (Tasks 2.1, 3.1 y 3.2).

In [3]:
class SinusoidalEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        if dim % 2 != 0:
            raise ValueError("dim debe ser par")
        frequencies = torch.exp(
            torch.arange(0, dim, 2, dtype=torch.float32)
            * (-math.log(10000.0) / dim)
        )
        self.dim = dim
        self.register_buffer("frequencies", frequencies)

    def forward(self, t):
        angles = t.float().unsqueeze(1) * self.frequencies.unsqueeze(0)
        embedding = torch.empty(t.size(0), self.dim, device=t.device)
        embedding[:, 0::2] = torch.sin(angles)
        embedding[:, 1::2] = torch.cos(angles)
        return embedding


class ConditionedBlock(nn.Module):
    def __init__(self, in_channels, out_channels, condition_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.condition_projection = nn.Linear(condition_dim, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.activation = nn.SiLU()

    def forward(self, x, condition):
        h = self.activation(self.norm1(self.conv1(x)))
        h = h + self.condition_projection(condition)[:, :, None, None]
        h = self.activation(self.norm2(self.conv2(h)))
        return h


class SmallConditionalUNet(nn.Module):
    def __init__(self, condition_dim=64):
        super().__init__()
        self.time_embedding = SinusoidalEmbedding(condition_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(condition_dim, condition_dim),
            nn.SiLU(),
            nn.Linear(condition_dim, condition_dim),
        )
        self.class_embedding = nn.Embedding(11, condition_dim)

        self.encoder_28 = ConditionedBlock(1, 32, condition_dim)
        self.downsample = nn.Conv2d(32, 64, 4, stride=2, padding=1)
        self.encoder_14 = ConditionedBlock(64, 64, condition_dim)
        self.bottleneck = ConditionedBlock(64, 64, condition_dim)

        self.upsample = nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1)
        self.decoder_28 = ConditionedBlock(64, 32, condition_dim)
        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x, t, c):
        time_condition = self.time_mlp(self.time_embedding(t))
        class_condition = self.class_embedding(c)
        condition = time_condition + class_condition

        skip = self.encoder_28(x, condition)
        h = self.downsample(skip)
        h = self.encoder_14(h, condition)
        h = self.bottleneck(h, condition)
        h = self.upsample(h)
        h = torch.cat([h, skip], dim=1)
        h = self.decoder_28(h, condition)
        return self.output(h)



T = 1000
betas = torch.linspace(1e-4, 0.02, T, device=device)    # calendario lineal beta_t
alphas = 1 - betas                                       # alpha_t = 1 - beta_t
alpha_bar = torch.cumprod(alphas, dim=0)                 # alpha_bar_t = prod alpha_s

X = (X01 * 2 - 1).unsqueeze(1)                           # difusión en [-1,1]
x_train, y_train = X[:-5000].to(device), Y[:-5000].to(device)

def q_sample(x0, t, eps):
    # x_t = sqrt(alpha_bar_t) x0 + sqrt(1 - alpha_bar_t) eps
    ab = alpha_bar[t - 1].view(-1, 1, 1, 1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

torch.manual_seed(SEED)
unet = SmallConditionalUNet().to(device)
optimizer = torch.optim.Adam(unet.parameters(), lr=2e-4)
ckpt = Path("data/task3_diffusion_checkpoint.pt")
train_losses = []
if ckpt.exists():
    saved = torch.load(ckpt, map_location=device)
    unet.load_state_dict(saved["model"]); optimizer.load_state_dict(saved["optimizer"])
    train_losses = saved["train_losses"]

t0 = time.perf_counter()
for epoch in range(len(train_losses), 30):
    unet.train()
    perm = torch.randperm(len(x_train), device=device)
    total = 0.0
    for i in range(0, len(x_train), 128):
        idx = perm[i:i + 128]
        x0, c = x_train[idx], y_train[idx].clone()
        t = torch.randint(1, T + 1, (len(x0),), device=device)
        eps = torch.randn_like(x0)
        c[torch.rand(len(x0), device=device) < 0.1] = 10   # condición nula con p_uncond
        loss = torch.mean((eps - unet(q_sample(x0, t, eps), t, c)) ** 2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(x0)
    train_losses.append(total / len(x_train))
    torch.save({"model": unet.state_dict(), "optimizer": optimizer.state_dict(),
                "train_losses": train_losses}, ckpt)
unet.eval()
train_time = time.perf_counter() - t0
print(f"Pérdida final: {train_losses[-1]:.5f} | tiempo de entrenamiento en esta sesión: {train_time:.1f} s")

Pérdida final: 0.04211 | tiempo de entrenamiento en esta sesión: 526.8 s


In [4]:
@torch.no_grad()
def ancestral_sample(model, labels):
    # Muestreo ancestral de la Task 3.3 (w = 1: solo predicción condicional)
    x = torch.randn(len(labels), 1, 28, 28, device=device)   # x_T ~ N(0, I)
    for step in range(T, 0, -1):
        t = torch.full((len(labels),), step, device=device, dtype=torch.long)
        eps_hat = model(x, t, labels)
        # x_{t-1} = 1/sqrt(alpha_t) (x_t - (1-alpha_t)/sqrt(1-alpha_bar_t) eps_hat) + sigma_t z
        mean = (x - (1 - alphas[step - 1]) / (1 - alpha_bar[step - 1]).sqrt() * eps_hat) / alphas[step - 1].sqrt()
        x = mean + betas[step - 1].sqrt() * torch.randn_like(x) if step > 1 else mean
    return x.clamp(-1, 1)

def sync():
    if device == "cuda":
        torch.cuda.synchronize()

# 500 muestras con w = 1 (50 por clase, como en la Task 3.4)
samples_ckpt = Path("data/task3_guided_w1.pt")
if samples_ckpt.exists():
    diff_500 = torch.load(samples_ckpt, map_location=device)["samples"].to(device)
else:
    torch.manual_seed(SEED)
    diff_500 = ancestral_sample(unet, torch.arange(10, device=device).repeat_interleave(50))
print(diff_500.shape)

torch.Size([500, 1, 28, 28])


**a.** Nitidez (Task 1.4 d) sobre 500 imágenes de cada fuente. Las muestras de difusión se pasan de [−1,1] a [0,1] para que la escala sea la misma que la del VAE y las reales.

In [5]:
def sharpness(imgs):
    # Magnitud media de la diferencia entre píxeles horizontalmente adyacentes
    imgs = imgs.reshape(-1, 28, 28)
    return (imgs[:, :, 1:] - imgs[:, :, :-1]).abs().mean().item()

torch.manual_seed(SEED)
with torch.no_grad():
    vae_500 = vae.dec(torch.randn(500, 2, device=device))   # z ~ N(0, I)
real_500 = X01[-5000:][:500]                                 # validación, [0,1]

sharp = {
    "Real": sharpness(real_500),
    "Difusión w=1": sharpness((diff_500 + 1) / 2),
    "VAE β=1": sharpness(vae_500),
}
for k, v in sharp.items():
    print(f"{k:14s} {v:.4f}")

Real           0.0886
Difusión w=1   0.0867
VAE β=1        0.0513


**b.** Tiempo para generar 100 imágenes y número de evaluaciones de red.

In [6]:
labels_100 = torch.arange(10, device=device).repeat_interleave(10)

# Calentamiento para no medir la inicialización de CUDA
with torch.no_grad():
    vae.dec(torch.randn(100, 2, device=device)); unet(torch.randn(100, 1, 28, 28, device=device), labels_100 + 1, labels_100)

torch.manual_seed(SEED); sync(); t0 = time.perf_counter()
with torch.no_grad():
    vae.dec(torch.randn(100, 2, device=device))          # 1 pasada del decoder
sync(); t_vae = time.perf_counter() - t0

torch.manual_seed(SEED); sync(); t0 = time.perf_counter()
ancestral_sample(unet, labels_100)                       # T = 1000 pasadas de la U-Net
sync(); t_diff = time.perf_counter() - t0

print(f"VAE:      {t_vae * 1000:.3f} ms | evaluaciones de red: 1")
print(f"Difusión: {t_diff:.3f} s    | evaluaciones de red: {T}")
print(f"Razón de tiempos difusión/VAE: {t_diff / t_vae:,.0f}x")

VAE:      0.365 ms | evaluaciones de red: 1
Difusión: 10.871 s    | evaluaciones de red: 1000
Razón de tiempos difusión/VAE: 29,755x


**c.** *(ver respuesta en la celda final)*

### Task 4.2

Estimador insesgado pass@k = 1 − C(n−c, k)/C(n, k) (vale 1 si n − c < k) y versión ingenua 1 − (1 − c/n)^k.

In [7]:
import pandas as pd
from math import comb

n = 20
problems = {"P1": 0, "P2": 2, "P3": 5, "P4": 9, "P5": 14, "P6": 20}

def pass_at_k(n, c, k):
    return 1.0 if n - c < k else 1 - comb(n - c, k) / comb(n, k)

def pass_naive(n, c, k):
    return 1 - (1 - c / n) ** k

tbl = pd.DataFrame({f"pass@{k}": {p: pass_at_k(n, c, k) for p, c in problems.items()} for k in (1, 5, 10)})
tbl["ingenua@5"] = {p: pass_naive(n, c, 5) for p, c in problems.items()}
tbl.loc["Promedio"] = tbl.mean()
print(tbl.round(4).to_string())

# b. P2: pass@k = 1 - (20-k)(19-k)/380; menor k con pass@k > 0.9
k_min = next(k for k in range(1, 21) if pass_at_k(n, 2, k) > 0.9)
assert all(abs(pass_at_k(n, 2, k) - (1 - (20 - k) * (19 - k) / 380)) < 1e-12 for k in range(1, 19))
print("Menor k con pass@k > 0.9 para P2:", k_min, "->", round(pass_at_k(n, 2, k_min), 4))

          pass@1  pass@5  pass@10  ingenua@5
P1        0.0000  0.0000   0.0000     0.0000
P2        0.1000  0.4474   0.7632     0.4095
P3        0.2500  0.8063   0.9837     0.7627
P4        0.4500  0.9702   0.9999     0.9497
P5        0.7000  0.9996   1.0000     0.9976
P6        1.0000  1.0000   1.0000     1.0000
Promedio  0.4167  0.7039   0.7911     0.6866
Menor k con pass@k > 0.9 para P2: 14 -> 0.9211


## Respuestas Task 4

### 4.1

a. Sobre 500 imágenes en escala $[0,1]$, la nitidez fue de $0.0886$ para las imágenes reales, $0.0867$ para la difusión con $w = 1$ y $0.0513$ para el VAE con $\beta = 1$. La difusión conserva cerca del 98 % de la nitidez de las imágenes reales, mientras que el VAE pierde alrededor del 42 %, lo cual coincide con lo observado en la Task 1.4(d).

b. Generar 100 imágenes con el VAE tomó $0.365$ ms y requirió una sola evaluación de red (una pasada del decoder), mientras que la difusión sin guía tomó $10.871$ s y requirió $1000$ evaluaciones de la U-Net, una por cada paso del muestreo ancestral. La razón de tiempos es de aproximadamente $29\,755$ veces a favor del VAE.

c. El VAE se ubica en el extremo de velocidad y estabilidad: genera en una sola pasada y su entrenamiento con la ELBO y la KL en forma cerrada converge sin problemas, pero sus muestras son borrosas ($0.051$ frente a $0.089$) porque la pérdida $L_2$ equivale a un decoder gaussiano con $\sigma_x^2 = \beta/2$, que promedia los modos de la distribución. La difusión se ubica en el extremo de nitidez y estabilidad: se entrena con un MSE simple sobre $\epsilon$ (pérdida final de $0.042$) y produce imágenes casi tan nítidas como las reales, pero es muy lenta porque necesita $1000$ pasos secuenciales. Para acortar su tiempo usaríamos muestreo DDIM con unos 50 pasos, un calendario coseno que desperdicie menos pasos en la región con $\mathrm{SNR} \ll 0.01$ (como se vio en la Task 2.5 b) y, a mayor plazo, destilación progresiva o consistency models para generar en uno a cuatro pasos.

### 4.2

a. Los promedios sobre los seis problemas son $\text{pass@}1 = 0.4167$, $\text{pass@}5 = 0.7039$ y $\text{pass@}10 = 0.7911$, con los valores por problema en la tabla anterior. Para P2 con $k = 5$ se tiene $\binom{18}{5} = \frac{18 \cdot 17 \cdot 16 \cdot 15 \cdot 14}{5!} = 8568$ y $\binom{20}{5} = \frac{20 \cdot 19 \cdot 18 \cdot 17 \cdot 16}{5!} = 15504$, de modo que el cociente es $\frac{8568}{15504} = 0.5526$ y $\text{pass@}5 = 1 - 0.5526 = 0.4474$.

b. Para P2 ($n = 20$, $c = 2$) se cumple $\frac{\binom{18}{k}}{\binom{20}{k}} = \frac{18!}{k!\,(18-k)!} \cdot \frac{k!\,(20-k)!}{20!} = \frac{(20-k)(19-k)}{20 \cdot 19}$, y como $20 \cdot 19 = 380$ se obtiene $\text{pass@}k = 1 - \frac{(20-k)(19-k)}{380}$. La condición $\text{pass@}k > 0.9$ equivale a $(20-k)(19-k) < 38$. Con $k = 13$ el producto es $42$ ($\text{pass@}13 = 0.889$) y con $k = 14$ es $30$ ($\text{pass@}14 = 0.921$), por lo que el menor valor es $k = 14$.

c. Para $k = 5$ la versión ingenua nunca supera al estimador insesgado. Para demostrarlo en general, si $n - c \ge k$ se escribe $\frac{\binom{n-c}{k}}{\binom{n}{k}} = \prod_{i=0}^{k-1} \frac{n-c-i}{n-i}$, y como cada factor cumple $\frac{n-c-i}{n-i} \le \frac{n-c}{n}$, el producto queda acotado por $\left(1 - \frac{c}{n}\right)^k$. Por lo tanto $1 - \frac{\binom{n-c}{k}}{\binom{n}{k}} \ge 1 - \left(1 - \frac{c}{n}\right)^k$. Si $n - c < k$, el estimador insesgado vale $1$, que es mayor o igual que cualquier probabilidad. La versión ingenua subestima porque supone muestreo con reemplazo, mientras que el estimador insesgado considera $k$ soluciones distintas tomadas sin reemplazo de las $20$ generadas.

d. Un agente que reintenta hasta 10 veces con un verificador perfecto tiene una probabilidad de éxito igual a $\text{pass@}10$, así que resolvería casi siempre P3, P4, P5 y P6 ($\text{pass@}10 \ge 0.98$), resolvería P2 solo en el 76 % de los casos y nunca resolvería P1, porque ninguna de sus 20 soluciones pasó las pruebas. El salto del promedio de $0.42$ en $\text{pass@}1$ a $0.79$ en $\text{pass@}10$ indica que el modelo suele tener una solución correcta dentro de su distribución, pero le asigna poca probabilidad en un solo intento. Muestrear varias veces y verificar casi duplica el rendimiento, aunque el techo queda en $5/6 \approx 0.83$, porque los reintentos no compensan la capacidad que le falta al modelo en P1.
